In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("LCEL-practice")

LangSmith 추적을 시작합니다.
[프로젝트명]
LCEL-practice


In [2]:
# cache
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path="./cache/LCEL.db"))

In [3]:
# runnablepassthrough랑 parallel 차이 보기
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    # 입력된 num 값에 x3 한 값을 내보내는 runnable설정
    extra=RunnablePassthrough.assign(mult=lambda x: x['num']*3),
    # 입력된 num 값에 +1 한 값을 내보내기
    modified=lambda x: x['num'] + 1,
)

answer = runnable.invoke({'num': 1})
answer

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [ ]:
# 실제 사용 사례는 생략


In [4]:
# runnable이 어떤 일이 일어나는지 파악하기
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    # 텍스트 데이터로부터 FAISS 벡터 저장소를 생성합니다.
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings(),
)

# 벡터 저장소를 기반으로 retriever를 생성합니다.
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}  

Question: {question}"""

prompt = ChatPromptTemplate.from_template(
    template
)  # 템플릿을 기반으로 ChatPromptTemplate을 생성합니다.

model = ChatOpenAI(model="gpt-4o-mini")  # ChatOpenAI 모델을 초기화합니다.

# chain 을 생성합니다.
chain = (
    # 검색 컨텍스트와 질문을 지정합니다.
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt  # 프롬프트를 생성합니다.
    | model  # 언어 모델을 실행합니다.
    | StrOutputParser()  # 출력 결과를 문자열로 파싱합니다.
)


In [5]:
# 체인의 그래프에서 노드를 가져오기
chain.get_graph().nodes

{'4139325011084a2c9e1606173f54fd75': Node(id='4139325011084a2c9e1606173f54fd75', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 '736aa870f7514de5b206025edaaacd56': Node(id='736aa870f7514de5b206025edaaacd56', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 'd34395d9b39e40fa843068dac4cdcd44': Node(id='d34395d9b39e40fa843068dac4cdcd44', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B9BFB904D0>, search_kwargs={}), metadata=None),
 '60d65920a1bf42db8ede65b20542ccb3': Node(id='60d65920a1bf42db8ede65b20542ccb3', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 '9b0dca94c7bc43fe9ae2143e7f258ebb': Node(id='9b0dca94c7bc43fe9ae2143e7f258ebb', name='ChatPromptTemplate', dat

In [6]:
# 체인의 그래프에서 엣지를 가져옵니다.
chain.get_graph().edges

[Edge(source='b01d025cf0fe4bc5b95a4c2e9c01e2b7', target='6d9129b7a3004ae4aeabb74f05dcff2e', data=None, conditional=False),
 Edge(source='6d9129b7a3004ae4aeabb74f05dcff2e', target='59028a6faeb440198dfe64ec4ae4e4da', data=None, conditional=False),
 Edge(source='b01d025cf0fe4bc5b95a4c2e9c01e2b7', target='13e4cac8367d4e4ab8b1cf96a697c8b4', data=None, conditional=False),
 Edge(source='13e4cac8367d4e4ab8b1cf96a697c8b4', target='59028a6faeb440198dfe64ec4ae4e4da', data=None, conditional=False),
 Edge(source='59028a6faeb440198dfe64ec4ae4e4da', target='c2dfa91f95ae4301ad978546643d5aac', data=None, conditional=False),
 Edge(source='c2dfa91f95ae4301ad978546643d5aac', target='79705d89aa904bd993c990cceeed8c34', data=None, conditional=False),
 Edge(source='8a4b03ebb8ef4afd8cc595befe455450', target='564a45eef9414cd0accecbe6f3a0d7fb', data=None, conditional=False),
 Edge(source='79705d89aa904bd993c990cceeed8c34', target='8a4b03ebb8ef4afd8cc595befe455450', data=None, conditional=False)]

In [7]:
# 체인의 그래프를 ASCII 형식으로 출력
# 보다 이해하기 쉬운 형태로 그래프 확인
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [8]:
# 체인에서 프롬프트 가져오기
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}  \n\nQuestion: {question}'), additional_kwargs={})])]

In [ ]:
# runnable lambda
# 사용자 정의 함수 실행
# 예를 들어 자신만의 함수 정의, 외부 api와의 상호 작용
# 사용자 정의 함수가 받을수 있는 인자는 "1개"

In [14]:
# 사용해보기
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

def length_function(text): # 텍스트 길이 반환 함수
    return len(text)

def _multiple_length_function(text1, text2): # 두 텍스트의 길이를 곱하는 함수
    return len(text1) * len(text2)

def multiple_length_function(_dict,): # 2개 인자를 받는 함수를 연결하는 wrapper 함수
    return _multiple_length_function(_dict['text1'], _dict['text2'])

# prompt
prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")
# chain1 = prompt | model

chain = (
    {"a": itemgetter("input_1") | RunnableLambda(length_function),
     "b": ({'text1': itemgetter("input_1"), 
           "text2": itemgetter("input_2")}
     
    | RunnableLambda(multiple_length_function)),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
answer = chain.invoke({'input_1': 'bar', 'input_2': 'gah'})
answer

'4 + 12 equals 16.'

In [ ]:
# surfapi, 테블렛search engine